<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/sample-answers/Bayesian_Inference_Assignment_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Sample Answer

### The 2PL Item Response Probability Model

Assuming that the item responses are conditionally independent given $\Theta=\theta$, the likelihood is

$$
P(Y=y\mid \Theta=\theta)=\prod_{i=1}^n
\left[
p_i(\theta)
\right]^{y_i}
\left[
1-p_i(\theta)
\right]^{1-y_i},
$$

where

$$
p_i(\theta)
=\frac{1}{1+e^{-a_i(\theta-b_i)}}.
$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
# Two distinct a_i values (0.5 and 1.5)
# For a_i = 1.5, we include 3 different b_i values (-2, 0, 2)
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

# Create the Plotly figure
fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]

    # Calculate probabilities
    p_vals = p_i(theta_vals, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

# Customize layout
fig.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# Display the interactive plot
fig.show()

### Posterior Distribution

At step $k$ (for $k = 1, \dots, n$), the current item response is $y_k$, and its likelihood contribution is:

$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

The sequential posterior density of $\Theta$ given the running response history vector $\mathbf{y}^{(k)}$ is updated recursively as follows:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) = \frac{ \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) }{ \int_{\mathbb R} \left[ p_k(s)^{y_k} (1 - p_k(s))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(s\mid\mathbf{y}^{(k-1)})\,ds }$$

**Definition of Terms:**

* $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$ is the **prior density** for the current step (which is the posterior inherited from the previous step).
* For the very first item ($k=1$), the base prior is the initialized distribution:

$$f_{\Theta\mid\mathbf{Y}^{(0)}}(\theta\mid\mathbf{y}^{(0)}) = f_\Theta^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{\theta^2}{2}\right)$$


* The denominator is the **local normalizing constant**, integrating out $\theta$ across the current likelihood-prior product to ensure the area under the new step-$k$ density curve integrates to 1.

### Baye's Estimate and the MAP Estimate

Under a sequential framework, point estimates like the **Posterior Mean** (Bayes estimate under squared-error loss) or the **Maximum A Posteriori (MAP)** estimate are updated dynamically at each step $k$ using the running posterior density $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

1. Running Posterior Mean (Bayes Estimate)
The Bayesian estimate of the user's latent ability at step $k$, minimizing the expected squared-error loss, is the expected value of the current posterior distribution:
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb E[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \int_{\mathbb R} \theta \, f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})\,d\theta$$

2. Running Maximum A Posteriori (MAP) Estimate
Alternatively, the most likely ability parameter at step $k$ corresponds to the mode (peak) of the current posterior density curve:
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} \in \arg\max_{\theta\in\mathbb R} f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$$

**Computation Note**

In practice, as the user answers more items, the computational grid tracks the updated array values for $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

* $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ is updated via a running vector dot-product: `np.trapz(theta * current_posterior, theta)`.
* $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ is extracted simply via the index of the maximum value: `theta[np.argmax(current_posterior)]`.

### Numerical Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# PART 1: SEQUENTIAL BAYESIAN UPDATE (MANUAL 4-ITEM SIMULATION)
# =====================================================================

# 1. Define a fine grid of latent ability values (theta)
theta = np.linspace(-5, 5, 500)

# 2. Initialize the prior distribution: Standard Normal N(0, 1)
prior = stats.norm.pdf(theta, 0, 1)

# 3. Define the Item Response Function (2PL Model)
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 4. Define the manual simulated sequence of item encounters
running_items = [
    {"a": 1.0, "b": -1.5, "y": 1},  # Step 1: Got an easy item correct
    {"a": 1.5, "b": 0.5,  "y": 1},  # Step 2: Got a medium-hard item correct
    {"a": 1.2, "b": 1.5,  "y": 0},  # Step 3: Got a very hard item incorrect
    {"a": 2.0, "b": 0.2,  "y": 1}   # Step 4: Got a highly discriminative item correct
]

# Create the first Plotly figure
fig1 = go.Figure()

# Add the initial Prior distribution trace
fig1.add_trace(go.Scatter(
    x=theta,
    y=prior,
    mode='lines',
    name='Initial Prior: N(0,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# Run the sequential update loop
current_posterior = prior.copy()

for idx, item in enumerate(running_items):
    a = item["a"]
    b = item["b"]
    y = item["y"]

    # Calculate item response probability across the grid
    prob = p_i(theta, a, b)

    # Calculate the likelihood contribution of this response
    likelihood = (prob ** y) * ((1 - prob) ** (1 - y))

    # Posterior proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerically normalize the density curve using the trapezoidal rule
    integral = np.trapezoid(current_posterior, theta)
    current_posterior /= integral

    # Format trace labels
    result_text = "Correct" if y == 1 else "Incorrect"
    trace_name = f"Step {idx+1}: After Item {idx+1} ({result_text}, a={a}, b={b})"

    # Add the current posterior step to the plot
    fig1.add_trace(go.Scatter(
        x=theta,
        y=current_posterior,
        mode='lines',
        name=trace_name,
        line=dict(width=2)
    ))

# Customize layout for Figure 1
fig1.update_layout(
    title={
        'text': "Sequential Bayesian Update of User Ability (θ)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the first plot
fig1.show()


# =====================================================================
# PART 2: PERFORMANCE TRACKING & CONVERGENCE TIMELINE (20 ITEMS)
# =====================================================================

# Set random seed for reproducibility of the random items/responses
np.random.seed(42)

# 1. Setup Simulation Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)  # Finer grid for tracking accuracy

# 2. Generate Random Item Characteristics
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# 3. Initialize Tracking Arrays (Step 0 = Initial Prior State)
running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

# Initialize prior density distribution: N(0, 1)
current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

# 4. Run the 20-item Sequential Simulation loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Compute true response probability at true theta
    prob_true = p_i(theta_true, a_k, b_k)

    # Simulate stochastically generated user response y_k
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    # Calculate likelihood curve across grid
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    # Update and normalize
    current_posterior_sim = current_posterior_sim * likelihood
    integral_sim = np.trapezoid(current_posterior_sim, theta_grid)
    current_posterior_sim /= integral_sim

    # Compute point estimates
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior_sim, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior_sim)]

    # Store estimates
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# 5. Create the second Plotly figure
fig2 = go.Figure()

# Add True Ability Reference Horizontal Line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Ability (θ = {theta_true})",
    annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

# Customize Layout for Figure 2
fig2.update_layout(
    title={
        'text': "Convergence of Latent Ability Estimators (θ) Over Time",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

# Display the second plot
fig2.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

### Visualization of the Beta Distribution

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# 1. Generate a dense grid of points over the bounded domain [0, 1]
theta_grid = np.linspace(0, 1, 500)

# 2. Define the three distinct parameter configurations
beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State: Beta(1,1)", "color": "gray", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State: Beta(2,8)", "color": "blue", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State: Beta(8,2)", "color": "green", "dash": "solid"}
]

# 3. Create the Plotly figure
fig = go.Figure()

for config in beta_configs:
    a = config["alpha"]
    b = config["beta"]

    # Calculate the exact analytical probability density function (PDF)
    pdf_vals = stats.beta.pdf(theta_grid, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_grid,
        y=pdf_vals,
        mode='lines',
        name=config["name"],
        line=dict(color=config["color"], dash=config["dash"], width=2.5)
    ))

# 4. Customize layout and design
fig.update_layout(
    title={
        'text': "Structural Variations of the Beta(α, β) Probability Density Function",
        'y': 0.93,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Parameter Value (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(range=[0, 1], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the interactive plot
fig.show()

### Sequential Likelihood and Joint History

**1. Single Isolated Response Likelihood**

At any specific impression step $k$, the user interaction $y_k \in \{0, 1\}$ is modeled as an independent Bernoulli trial conditional on the true click probability $\theta$. The mathematical likelihood contribution $L(y_k \mid \theta)$ of this single outcome is expressed as:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

* If the user clicks ($y_k = 1$), the expression simplifies to $\theta^1 (1-\theta)^0 = \theta$.
* If the user does not click ($y_k = 0$), the expression simplifies to $\theta^0 (1-\theta)^1 = 1 - \theta$.

**2. Joint Likelihood Function for the Running History**

Assuming that individual user interactions are conditionally independent given the underlying conversion parameter $\Theta = \theta$, the joint likelihood function for the entire running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual step likelihoods up to time step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{\sum_{i=1}^k (1 - y_i)}$$

Letting $C_k = \sum_{i=1}^k y_i$ represent the total running count of clicks, and $N_k - C_k = \sum_{i=1}^k (1 - y_i)$ represent the total running count of non-clicks (where $N_k = k$ is the total number of impressions), the joint history likelihood simplifies compactly to:

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{C_k} (1 - \theta)^{k - C_k}$$

---

**Closed-Form Analytical Updates (Conjugacy)**

**1. Recursive Relationship via Bayes' Theorem**

In a running update framework, the posterior distribution at step $k-1$ acts directly as the prior distribution for step $k$. Applying Bayes' Theorem sequentially, the posterior density after observing the newest single event $y_k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid s) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)})\,ds}$$

Dropping the denominator (the local normalizing constant) yields the recursive formulation up to a proportionality constant:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

**2. Analytical Proof of Beta-Binomial Conjugacy**

Assume that the prior density at the start of step $k$ (which is the posterior inherited from step $k-1$) belongs to the Beta family with shape parameters $\alpha_{k-1}$ and $\beta_{k-1}$:

$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \propto \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$

Substituting the single isolated response likelihood from Task 2 into the proportional recursive framework:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Combining like algebraic bases by summing their exponential powers:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} \cdot (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

Because the resulting kernel $\theta^{\text{power}_1 - 1}(1-\theta)^{\text{power}_2 - 1}$ matches the exact structural form of a Beta probability density function, the posterior is proven to remain inside the Beta family.

**3. Closed-Form Arithmetic Update Parameters**

Equating the exponent powers directly reveals that the new analytical state parameters $(\alpha_k, \beta_k)$ are simple deterministic linear updates of the previous state:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Including the exact beta normalization constant, the explicit closed-form posterior density at step $k$ is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{1}{\mathrm{B}(\alpha_k, \beta_k)} \theta^{\alpha_k - 1} (1 - \theta)^{\beta_k - 1}$$

### Posterior Mean


In the context of the Beta-Binomial conjugate update problem, it has a clean, exact analytical solution. Since the posterior distribution at step $k$ is $\text{Beta}(\alpha_k, \beta_k)$, the expected value is given by the standard formula for a Beta distribution's mean:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

### Expanding via the Closed-Form Updates

Using the recursive arithmetic update rules we derived in Task 3:

* $\alpha_k = \alpha_0 + \sum_{i=1}^k y_i = \alpha_0 + C_k$ *(where $C_k$ is the total number of clicks observed up to step $k$)*
* $\beta_k = \beta_0 + \sum_{i=1}^k (1 - y_i) = \beta_0 + (k - C_k)$ *(where $k - C_k$ is the total number of non-clicks)*

Substituting these updates directly into the expectation formula gives:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_0 + C_k}{(\alpha_0 + \beta_0) + k}$$

**Interpretation of the Result:**

This closed-form solution has a beautiful intuitive structure in data analysis:

1. **Weighted Average:** The posterior mean behaves as a weighted average combining the initial prior beliefs ($\alpha_0, \beta_0$) with the real-world sample evidence ($C_k$ successes out of $k$ total impressions).
2. **Asymptotic Convergence:** As the number of impressions grows very large ($k \to \infty$), the constants from the prior ($\alpha_0$ and $\beta_0$) become mathematically insignificant. The formula collapses directly to $\frac{C_k}{k}$, which is the empirical sample proportion of clicks (the maximum likelihood estimate).

### Numerical Simulation

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(42)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.35  # The true click-through rate of the advertisement (35%)
n_impressions = 100
steps = list(range(n_impressions + 1))

# Initialize Beta Prior Parameters (Prior: Uniform distribution Beta(1,1))
alpha_param = 1
beta_param = 1

# Setup grid for plotting probability density curves (Domain: 0 to 1)
theta_grid = np.linspace(0, 1, 500)

# Define specific milestone steps where we want to capture the full curve shape
milestones = [0, 1, 2, 5, 10, 30, 50, 100]

# Tracking arrays for point estimates
running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]  # Mode of Uniform(0,1)

# Initialize Figure 1 for the probability density progression curves
fig1 = go.Figure()

# Plot the initial Prior Beta(1, 1) distribution curve
prior_density = stats.beta.pdf(theta_grid, alpha_param, beta_param)
fig1.add_trace(go.Scatter(
    x=theta_grid, y=prior_density, mode='lines',
    name='Initial Prior: Beta(1,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# RUNNING ANALYTICAL UPDATE LOOP
# =====================================================================
for k in range(1, n_impressions + 1):
    # Simulate a user action stochastically based on true CTR
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # EXACT CLOSED-FORM CONJUGATE UPDATE RULES:
    alpha_param += y_k
    beta_param += (1 - y_k)

    # Calculate exact point estimates directly from analytical formulas
    theta_bayes_k = alpha_param / (alpha_param + beta_param)

    # Guard formula for mode when alpha or beta <= 1
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

    # If the current step hits one of our milestone checkpoints, capture its PDF
    if k in milestones:
        # Evaluate exact density curve across the grid analytically
        density_k = stats.beta.pdf(theta_grid, alpha_param, beta_param)
        result_text = "Click" if y_k == 1 else "No Click"

        fig1.add_trace(go.Scatter(
            x=theta_grid, y=density_k, mode='lines',
            name=f"Step {k}: After Event ({result_text}, α={alpha_param}, β={beta_param})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE FIGURE 1: POSTERIOR DISTRIBUTION PROGRESSION
# =====================================================================
# Add True CTR vertical line to see where the density curves are peaking
fig1.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2,
    annotation_text=f"True CTR ({theta_true})", annotation_position="top right"
)

fig1.update_layout(
    title={
        'text': "Analytical Posterior Density Progression (Beta-Binomial Updates)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Conversion Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
fig1.show()

# =====================================================================
# VISUALIZE FIGURE 2: CONVERGENCE TIMELINE
# =====================================================================
fig2 = go.Figure()

# Add True CTR Reference Line
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ = {theta_true})", annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes, mode='lines',
    name='Exact Posterior Mean (Beta Formula)',
    line=dict(color='blue', width=2.5)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map, mode='lines',
    name='Exact MAP Estimate (Beta Formula)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig2.update_layout(
    title={
        'text': "Analytical Beta-Binomial Conjugate Update Timeline",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of User Impressions (k)",
    yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)
fig2.show()

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Sample Answer

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(24)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.68       # True remaining stiffness efficiency of the beam (68%)
K_nominal = 50.0        # Nominal baseline stiffness of the pristine structure (kN/mm)
sigma = 0.15            # Sensor noise standard deviation (log-space)
n_sensor_readings = 15  # Timeline steps

# 1. Define a fine grid over the physical boundary [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 500)

# 2. Initialize Prior: Bounded Beta distribution reflecting an initially healthy beam
# centered heavily near 0.95-1.0
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
# Normalize initial prior
current_posterior /= np.trapezoid(current_posterior, theta_grid)

# Steps milestone tracking for plotting curves
milestones = [0, 1, 2, 5, 10, 15]

# Create Figure
fig = go.Figure()

# Plot Initial Prior State
fig.add_trace(go.Scatter(
    x=theta_grid, y=current_posterior, mode='lines',
    name='Prior State: Structural Health Assumed Healthy',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# SEQUENTIAL BAYESIAN MONITORING LOOP
# =====================================================================
for k in range(1, n_sensor_readings + 1):
    # Simulate a noisy structural sensor reading from log-normal physics
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    # Calculate Log-Normal Likelihood curve across the structural theta grid
    # Expected value for any grid point is: grid_point * K_nominal
    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    # Running Update: Posterior Proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerical Normalization via Trapezoidal rule
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral

    # Capture structural health density profile at milestones
    if k in milestones:
        fig.add_trace(go.Scatter(
            x=theta_grid, y=current_posterior, mode='lines',
            name=f"Step {k}: Post-Sensor Reading (Observed K={y_k:.2f})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE STRUCTURAL DEGRADATION TRACKING
# =====================================================================
fig.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2.5,
    annotation_text=f"True Structural Degradation State ({theta_true})",
    annotation_position="top left"
)

fig.update_layout(
    title={
        'text': "Structural Health Monitoring: Bounded Bayesian Parameter Updating",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Remaining Structural Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density (Confidence level of damage)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

fig.show()

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster (k).

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of (X_i) is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation (x_i), use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$

This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\
\gamma_{i2}\
\vdots\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation

In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility (\gamma_{ik}) acts as a fractional membership weight of observation (x_i) in cluster (k).

---

9. Interpretation

Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.

Your answer should mention the following points:

1. The mixture weight (\phi_k) is the prior probability of cluster (k).
2. The Gaussian density (\mathscr N(x_i\mid \mu_k,\Sigma_k)) measures how compatible (x_i) is with cluster (k).
3. The responsibility (\gamma_{ik}) is the posterior probability of cluster (k) after observing (x_i).
4. The soft assignment vector is

$$
\mathbb E[Z_i\mid X_i=x_i].
$$

5. The M-step updates the cluster parameters using these posterior membership probabilities as weights.

Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.
